In [ ]:
# ---- which sweeps "Run All" runs: True = run, False = skip -------------------------------------
# every sweep cell below checks its flag; a staged sweep runs all its stages in order, each on top of the best values
# of the stages before. Run time per run on the RTX 5080 where measured
sweep_bias_model = False                        # 100 runs, a few seconds each (done 8 Nov)
sweep_svd = False                               # 4 runs, ~80 s each (done 8 Nov)
sweep_svdpp = False                             # 4 runs, ~6 min each
sweep_factorization_machine = True              # 15 runs in 4 stages, ~2.5 min each (k = 250: ~2x)
sweep_item_knn = False                          # 108 runs, ~20 s each (CPU; n_neighbors done 8 Nov)
sweep_integrated_model = True                   # 13 runs in 3 stages, SVD++-sized (k = 500): roughly 10 min each
sweep_autorec = True                            # 10 runs in 2 stages
sweep_rating_transformer = True                 # 14 runs in 3 stages
sweep_bayesian_factorization_machine = True     # 7 runs in 3 stages, CPU, by far the slowest (hours): ~6 s per draw at rank 32

# Task 1 hyperparameter sweeps

**Run All:** set the flags in the first cell, then run the whole notebook. Every sweep cell only runs if its flag is `True`.
A staged sweep (factorization machine and the new models) runs all its stages in order; every stage passes the best values
of the stages before on as constants (`best_values_of`), so it does not depend on copying values into `lib/config.py` in
between. Run a single staged cell by hand only after the stages before it (it needs their `..._best_values`).
A sweep that fails stops Run All there; its finished runs are in its output file.

`run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 1 section)

Every run trains on the 90 % training split and is scored on the 10 % validation split (validation MSE).
No refit on 100 %, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task1/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py` as it is), the swept and constant values and the score of every run.
For a staged sweep: copy `best.parameters` of its last stage (before a noise check) - it contains the earlier stages' best values.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** `rerun_finalists_on_other_splits=True` re-trains the best `n_finalists` (default 3) combinations on the
validation splits with seeds 1 and 2; the winner is then the best mean over the three splits (2 extra runs per finalist).
`show_training_log=True` prints the epoch lines of every run.

Run time per run on the RTX 5080 (Task 1 sweep, 7 Nov): SVD (k = 120) ~80 s, SVD++ (k = 500) ~6 min, bias model a few seconds;
factorization machine (k = 120, 14 Nov) ~1 - 2 min at batch 512, ~2 - 4 min at batch 256.

In [1]:
%load_ext autoreload
%autoreload 2

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import best_values_of, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


## Bias model (closed form)

In [9]:
if sweep_bias_model:
    # a few seconds per run
    bias_model_sweep = run_grid_search("task1", "bias_model", {
        "user_bias_regularization": [0.5, 1.0, 2.0, 3.0],
        "item_bias_regularization": [3.0, 5.0, 8.0, 16.0, 24.0],
        "n_iterations": [10, 20, 30, 50, 100],
    })

task1 / bias_model: 100 combinations | swept: {'user_bias_regularization': [0.5, 1.0, 2.0, 3.0], 'item_bias_regularization': [3.0, 5.0, 8.0, 16.0, 24.0], 'n_iterations': [10, 20, 30, 50, 100]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\bias_model_2026-11-08_06-58-26.json
[1/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=10 ...
    validation MSE 0.76321 | 0 s | best so far 0.76321
[2/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=20 ...
    validation MSE 0.76306 | 0 s | best so far 0.76306
[3/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=30 ...
    validation MSE 0.76303 | 0 s | best so far 0.76303
[4/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=50 ...
    validation MSE 0.76301 | 1 s | best so far 0.76301
[5/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=100 ...
    validat

## SVD

In [3]:
if sweep_svd:
    # ~80 s per run (k = 120)
    svd_sweep = run_grid_search("task1", "svd", {
        "user_bias_regularization": [1.0, 3.0],
        "item_bias_regularization": [3.0, 5.0],
        "n_factors": 120,
    })

task1 / svd: 4 combinations | swept: {'user_bias_regularization': [1.0, 3.0], 'item_bias_regularization': [3.0, 5.0]} | constant: {'n_factors': 120}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\svd_2026-11-08_06-37-31.json
[1/4] user_bias_regularization=1 item_bias_regularization=3 ...
    validation MSE 0.74895 (epoch 7) | 84 s | best so far 0.74896
[2/4] user_bias_regularization=1 item_bias_regularization=5 ...
    validation MSE 0.74856 (epoch 7) | 82 s | best so far 0.74856
[3/4] user_bias_regularization=3 item_bias_regularization=3 ...
    validation MSE 0.75021 (epoch 7) | 82 s | best so far 0.74856
[4/4] user_bias_regularization=3 item_bias_regularization=5 ...
    validation MSE 0.74975 (epoch 7) | 83 s | best so far 0.74856
best: user_bias_regularization=1 item_bias_regularization=5 -> validation MSE 0.74856


## SVD++

In [ ]:
if sweep_svdpp:
    # ~6 min per run (k = 500) -> 4 runs ~25 min; with the finalist check (2 finalists x 2 splits) ~50 min
    svdpp_sweep = run_grid_search("task1", "svdpp", {
        "user_bias_regularization": [1.0, 3.0],
        "item_bias_regularization": [3.0, 5.0],
    }, rerun_finalists_on_other_splits=False, n_finalists=2)

## Factorization machine
Small grids, the most important parameters first, on top of `TASK1_FACTORIZATION_MACHINE` in `lib/config.py`:

1. **Optimisation** - done 14 Nov: `learning_rate` 2.5e-4 / 5e-4 / 1e-3 x `ratings_per_batch` 256 / 512 -> 5e-4 / 256 best
   (0.7515; the old 2.5e-4 / 512: 0.7553), now in `lib/config.py`
2. **Capacity and regularisation**: `n_factors` x `factor_regularization`
3. **Smaller effects**, one at a time: `bias_regularization`, then `n_activity_buckets`
4. **Noise check**: the final settings with 3 seeds -> differences smaller than their spread are noise

Earlier sweeps (8 Nov): `factor_regularization` 4 -> 0.7567, 8 -> 0.7553, 16 -> 0.7552 (k = 120);
at k = 20, `ratings_per_batch` 256 beat 512 by ~0.0015 and 5 / 10 / 20 buckets were within 0.0005.
Not worth sweeping: `max_epochs` (early stopping ends every run long before), `early_stopping_patience`.
Longer sweeps later: `rerun_finalists_on_other_splits=True` checks the best settings on two more validation splits.

In [ ]:
if sweep_factorization_machine:
    # stage 2 - capacity x regularisation (6 runs; k = 250 takes ~2x as long as k = 120)
    factorization_machine_capacity_sweep = run_grid_search("task1", "factorization_machine", {
        "n_factors": [120, 250],
        "factor_regularization": [8.0, 16.0, 32.0],
    })
    factorization_machine_best_values = best_values_of(factorization_machine_capacity_sweep)

In [ ]:
if sweep_factorization_machine:
    # stage 3a - one lambda for b_u and b_i (also the closed-form starting biases) (3 runs)
    factorization_machine_bias_sweep = run_grid_search("task1", "factorization_machine", {
        **factorization_machine_best_values,
        "bias_regularization": [1.0, 3.0, 5.0],
    })
    factorization_machine_best_values |= best_values_of(factorization_machine_bias_sweep)

In [ ]:
if sweep_factorization_machine:
    # stage 3b - number of user activity / item popularity buckets (3 runs)
    factorization_machine_bucket_sweep = run_grid_search("task1", "factorization_machine", {
        **factorization_machine_best_values,
        "n_activity_buckets": [5, 10, 20],
    })
    factorization_machine_best_values |= best_values_of(factorization_machine_bucket_sweep)

In [ ]:
if sweep_factorization_machine:
    # stage 4 - noise check: the best settings of the stages before with 3 seeds (initial factors + batch order)
    # -> differences between settings smaller than this spread are noise
    factorization_machine_seed_check = run_grid_search("task1", "factorization_machine", {
        **factorization_machine_best_values,
        "seed": [0, 1, 2],
    })

## Item-item kNN

In [ ]:
if sweep_item_knn:
    # CPU, ~20 s per run (neighbourhoods of all items)
    item_knn_sweep = run_grid_search("task1", "item_knn", {
        "n_neighbors": [25, 50, 100],
        "similarity_shrinkage": [80.0, 100.0, 120.0],
        "max_ratings_per_user_for_similarity": [100, 200, 300, 400],
        "items_per_similarity_block": [1000, 2000, 3000],
    })

task1 / item_knn: 3 combinations | swept: {'n_neighbors': [25, 50, 100]} | constant: {'similarity_shrinkage': 100.0}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\item_knn_2026-11-08_06-47-39.json
[1/3] n_neighbors=25 ...
    validation MSE 0.75658 | 19 s | best so far 0.75658
[2/3] n_neighbors=50 ...
    validation MSE 0.75636 | 19 s | best so far 0.75636
[3/3] n_neighbors=100 ...
    validation MSE 0.75615 | 19 s | best so far 0.75615
best: n_neighbors=100 -> validation MSE 0.75615


## New models (14 Nov): integrated model, I-AutoRec, rating transformer, Bayesian FM
Staged like the factorization machine: the stages of a model run in order, each on top of the best values of the stages before.
The Bayesian FM comes last: it runs on the CPU and is by far the slowest.

First check on the real Task 1 data (CPU, small sizes, no refit; SVD++ with k = 32: 0.7506): Bayesian FM 0.7255,
integrated model 0.7366, AutoRec 0.7506, rating transformer 0.7590 after 4 epochs (still falling). Details in `lib/config.py`.

### Integrated model (GPU; the neighbour lookups run on the CPU, ~20 s per epoch at batch 512)
Starts from the tuned SVD++ (k = 500, learning rate 2.5e-4, batch 512) and the item kNN's 100 neighbours per book.
1. **Neighbourhood**: `n_neighbors` x `neighbour_regularization` (the parts SVD++ does not have)
2. **Factors**: `n_factors` x `factor_regularization`
3. **Learning rate**

In [ ]:
if sweep_integrated_model:
    # stage 1 - neighbours per book x their penalty (6 runs)
    integrated_model_neighbourhood_sweep = run_grid_search("task1", "integrated_model", {
        "n_neighbors": [100, 200],
        "neighbour_regularization": [0.1, 1.0, 10.0],
    })
    integrated_model_best_values = best_values_of(integrated_model_neighbourhood_sweep)

In [ ]:
if sweep_integrated_model:
    # stage 2 - factors x their penalty (4 runs; k = 500 takes ~2x as long as k = 250)
    integrated_model_factor_sweep = run_grid_search("task1", "integrated_model", {
        **integrated_model_best_values,
        "n_factors": [250, 500],
        "factor_regularization": [8.0, 16.0],
    })
    integrated_model_best_values |= best_values_of(integrated_model_factor_sweep)

In [ ]:
if sweep_integrated_model:
    # stage 3 - learning rate (3 runs)
    integrated_model_learning_rate_sweep = run_grid_search("task1", "integrated_model", {
        **integrated_model_best_values,
        "learning_rate": [1.25e-4, 2.5e-4, 5e-4],
    })

### I-AutoRec (GPU; on the residuals of the bias model, masked training)
1. **Optimisation and regularisation**: `learning_rate` x `regularization` (the CPU checks: learning rate 1e-3 made it worse
   than the bias model from the first epoch on; 1e-4 with lambda 16 improved steadily)
2. **Capacity and masking**: `hidden_dimension` x `masked_fraction` (share of a book's ratings hidden from its input per batch)

In [ ]:
if sweep_autorec:
    # stage 1 - learning rate x lambda (6 runs)
    autorec_optimisation_sweep = run_grid_search("task1", "autorec", {
        "learning_rate": [3e-5, 1e-4, 3e-4],
        "regularization": [16.0, 64.0],
    })
    autorec_best_values = best_values_of(autorec_optimisation_sweep)

In [ ]:
if sweep_autorec:
    # stage 2 - hidden units x masked share (4 runs)
    autorec_capacity_sweep = run_grid_search("task1", "autorec", {
        **autorec_best_values,
        "hidden_dimension": [250, 500],
        "masked_fraction": [0.25, 0.5],
    })

### Rating transformer (GPU)
1. **Optimisation**: `learning_rate` x `dropout` (transformers are sensitive to both)
2. **Capacity**: `embedding_dimension` x `n_blocks`
3. **History**: `max_history_length` x `use_test_pairs_in_history` (the test pairs as extra history tokens with the rating "unknown")

In [ ]:
if sweep_rating_transformer:
    # stage 1 - learning rate x dropout (6 runs)
    rating_transformer_optimisation_sweep = run_grid_search("task1", "rating_transformer", {
        "learning_rate": [5e-4, 1e-3, 2e-3],
        "dropout": [0.1, 0.3],
    })
    rating_transformer_best_values = best_values_of(rating_transformer_optimisation_sweep)

In [ ]:
if sweep_rating_transformer:
    # stage 2 - width x depth (4 runs; d = 128 / 2 blocks are the slowest)
    rating_transformer_capacity_sweep = run_grid_search("task1", "rating_transformer", {
        **rating_transformer_best_values,
        "embedding_dimension": [64, 128],
        "n_blocks": [1, 2],
    })
    rating_transformer_best_values |= best_values_of(rating_transformer_capacity_sweep)

In [ ]:
if sweep_rating_transformer:
    # stage 3 - history length x test pairs in the history (4 runs; attention cost grows with the square of the length)
    rating_transformer_history_sweep = run_grid_search("task1", "rating_transformer", {
        **rating_transformer_best_values,
        "max_history_length": [100, 200],
        "use_test_pairs_in_history": [False, True],
    })

### Bayesian factorization machine (CPU, `pip install myfm`; the slowest sweep -> last)
No learning rate, lambda or early stopping: only the size of the model and the number of Gibbs draws.
1. **Rank** at a short chain (20 + 80 draws): a Bayesian model rarely overfits, larger ranks usually help
2. **Number of kept draws** at the best rank: more draws = a smoother posterior mean (slower, never worse in expectation)
3. **Implicit feedback from the test pairs**: N(u) / N(i) also from the pairs of `pairs_Rating.csv` (no gain for SVD++)

Time per draw in the CPU check: rank 32 ~6 s, rank 64 ~12 s (about linear in the rank) -> stage 1 alone ~1 h.

In [ ]:
if sweep_bayesian_factorization_machine:
    # stage 1 - rank (3 runs) at a short chain
    bayesian_fm_rank_sweep = run_grid_search("task1", "bayesian_factorization_machine", {
        "rank": [32, 64, 128],
        "n_burn_in_iterations": 20,
        "n_kept_samples": 80,
    })
    bayesian_fm_best_values = best_values_of(bayesian_fm_rank_sweep)

In [ ]:
if sweep_bayesian_factorization_machine:
    # stage 2 - kept draws at the best rank (2 runs; burn-in as in lib/config.py)
    bayesian_fm_draws_sweep = run_grid_search("task1", "bayesian_factorization_machine", {
        **bayesian_fm_best_values,
        "n_kept_samples": [180, 380],
    })
    bayesian_fm_best_values |= best_values_of(bayesian_fm_draws_sweep)

In [ ]:
if sweep_bayesian_factorization_machine:
    # stage 3 - N(u) / N(i) from the training ratings vs. from every known pair incl. the test pairs (2 runs)
    bayesian_fm_test_pairs_sweep = run_grid_search("task1", "bayesian_factorization_machine", {
        **bayesian_fm_best_values,
        "use_test_pairs_as_implicit_feedback": [False, True],
    })

## Overview

In [6]:
# one line per sweep file in output/task1 (also works right after a kernel restart)
print_sweep_overview("task1")

bias_model_2026-11-08_06-36-37.json                finished         12 / 12 | best 0.76284 with user_bias_regularization=1 item_bias_regularization=5
factorization_machine_2026-11-08_06-43-13.json     finished           3 / 3 | best 0.75520 with factor_regularization=16
item_knn_2026-11-08_06-47-39.json                  finished           3 / 3 | best 0.75615 with n_neighbors=100
svd_2026-11-08_06-37-31.json                       finished           4 / 4 | best 0.74856 with user_bias_regularization=1 item_bias_regularization=5
